# 06 - Fine-tuning (LoRA)
One notebook for all training runs. Only `variant` changes:

| variant | training file | model saved in |
|---|---|---|
| `cot` | sft_cot.json | models/qwen2b-cot |
| `refine` | sft_refine.json | models/qwen2b-refine |
| `review` | sft_review.json | models/qwen2b-review |
| `review_27b` | sft_review_27b.json | models/qwen2b-review-27b |

Run on Kaggle (1x T4), about 4 to 5 hours per run.

In [ ]:
!pip uninstall -y torchao
!pip install -U transformers==5.18.0 peft==0.21.1 trl==1.14.1 datasets accelerate

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="0"

In [ ]:
import sys
import torch
import transformers
import peft
import datasets
import trl

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Datasets:", datasets.__version__)
print("TRL:", trl.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
variant='cot'          # 'cot' | 'review' | 'refine'
max_steps=-1           # -1 = full run, small number = quick test

In [ ]:
import json
import glob

dataset_path=glob.glob(f'/kaggle/input/**/sft_{variant}.json',recursive=True)[0]
print(dataset_path)

with open(dataset_path,'r',encoding='utf-8') as file:
    records=json.load(file)

print(len(records))
records[0]

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name='Qwen/Qwen3.5-2B-Base' # 4B run: 'Qwen/Qwen3.5-4B-Base'
model_tag='qwen2b'                 # 4B run: 'qwen4b'
train_dtype=torch.float32          # 4B run: torch.float16 (float32 weights of 4B do not fit on one T4)

tokenizer=AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token=tokenizer.eos_token

print("Tokenizer loaded")
print("EOS Token:", tokenizer.eos_token)

In [ ]:
def format_prompt(record: dict)->str:
    return '[Question] '+record.get('question')+'\n'

def format_completion(record: dict, eos: str)->str:
    result=""

    for i in range(1,len(record.get('chains'))+1):
        result+=f'[Answer {i}] \n'+record.get('chains')[i-1]+'\n'

    if record.get('review')!=None:
        result+='[Review] \n'+record.get('review')+'\n'

    if record.get('refined')!=None:
        result+='[Refined answer] \n'+record.get('refined')+'\n'

    result+='[Final answer] '+record.get('final_answer')

    result+=eos

    return result

In [ ]:
print(format_prompt(records[0])+format_completion(records[0],tokenizer.eos_token))

In [ ]:
from datasets import Dataset

train_rows=[{'prompt': format_prompt(i), 'completion': format_completion(i,tokenizer.eos_token)} for i in records]

assert all(r['completion'].endswith(tokenizer.eos_token) for r in train_rows)

lengths=[len(tokenizer(r['prompt']+r['completion'])['input_ids']) for r in train_rows]
print('min', min(lengths))
print('max', max(lengths))
print('avg', sum(lengths)/len(lengths))

train_rows=[r for r,l in zip(train_rows,lengths) if l<=2048]
dataset=Dataset.from_list(train_rows)
print(dataset)

In [ ]:
base_model=AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=train_dtype,
    device_map={"": 0},
)

print("Model loaded")
print(type(base_model).__name__)

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config=LoraConfig(
    r=16,
    lora_alpha=32,
    bias="none",
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules=r"^(?!.*(visual|mtp)).*layers\.\d+\.(self_attn\.(q|k|v|o)_proj|linear_attn\.(in_proj_qkv|in_proj_z|out_proj)|mlp\.(gate|up|down)_proj)$",
)

peft_model=get_peft_model(base_model,lora_config)
peft_model.print_trainable_parameters()

In [ ]:
from trl import SFTConfig

training_args=SFTConfig(
    output_dir=f'/kaggle/working/models/{model_tag}-{variant}',
    num_train_epochs=2,
    max_steps=max_steps,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    weight_decay=0.0,
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    fp16=True,
    bf16=False,
    max_length=2048,
    completion_only_loss=True,
    logging_strategy="steps",
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
)

print("Training config created")

In [ ]:
from trl import SFTTrainer

trainer=SFTTrainer(
    model=peft_model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("SFT trainer created")

In [ ]:
sample=trainer.train_dataset[0]
print(tokenizer.decode(sample['input_ids'][-10:]))
print("Ends with EOS:", sample['input_ids'][-1]==tokenizer.eos_token_id)
print(sample.keys())

In [ ]:
trainer.train()

In [ ]:
print("Done training")

In [ ]:
final_output=f'/kaggle/working/models/{model_tag}-{variant}-final'

trainer.save_model(final_output)
tokenizer.save_pretrained(final_output)

print("Final model saved to:", final_output)

In [ ]:
import shutil

shutil.make_archive(final_output,"zip",root_dir=final_output)

print("Created:", final_output+".zip")